<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sinhala-wsd/blob/sinhala_textbook_extraction/textbook_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Setup

*   Installs pdftotext (to read the PDF’s text) and pandukabhaya (to convert FM-Abhaya to Unicode), then connects Drive.

In [2]:
!apt-get -qq update > /dev/null && apt-get -qq install -y poppler-utils > /dev/null
!pip install -q pandukabhaya

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, random, subprocess, unicodedata, datetime
from collections import Counter
from pandukabhaya import Converter

print(subprocess.run(["pdftotext", "-v"], capture_output=True, text=True).stderr.splitlines()[0])

Mounted at /content/drive
pdftotext version 24.02.0


2. Load the book

In [5]:
BASE    = "/content/drive/MyDrive/Research-WSD-New"
PDF_DIR = f"{BASE}/textbooks"
OUT_DIR = f"{BASE}/textbooks/Ext_Final"

BOOKS = [
    {
        "book_id":     "sinhala_lit_g10_11",
        "pdf":         "sinhala lit g-10-11.pdf",
        "title":       "සිංහල සාහිත්‍ය සංග්‍රහය (Grade 10–11)",
        "publisher":   "Educational Publications Department",
        "grade":       "10-11",
        "year":        "",
        "url":         "",
        "page_offset": 10,
    },
]
for b in BOOKS:
    b["pdf_path"] = os.path.join(PDF_DIR, b["pdf"])
    b["out"] = os.path.join(OUT_DIR, b["book_id"]); os.makedirs(b["out"], exist_ok=True)
    print(("OK      " if os.path.exists(b["pdf_path"]) else "MISSING ") + b["pdf_path"])

OK      /content/drive/MyDrive/Research-WSD-New/textbooks/sinhala lit g-10-11.pdf


3. Read the text and check its encoding

In [6]:
SINHALA = re.compile(r"[\u0D80-\u0DFF]")
LATIN   = re.compile(r"[A-Za-z]")

def pdf_pages(path):
    """pdftotext output split into pages (form feed = page break). Index 0 = PDF page 1."""
    raw = subprocess.run(["pdftotext", path, "-"], capture_output=True, text=True, check=True).stdout
    pages = raw.split("\f")
    return pages[:-1] if pages and not pages[-1].strip() else pages

def detect_encoding(pages):
    """'unicode' = already Sinhala Unicode; 'fm' = legacy font (Latin letters stand for Sinhala glyphs);
    'none' = no text layer (scanned) -> needs OCR (Document AI) instead."""
    text = "".join(pages)
    s, l = len(SINHALA.findall(text)), len(LATIN.findall(text))
    if s + l < 500:
        return "none", s, l
    return ("unicode" if s > l else "fm"), s, l

for b in BOOKS:
    b["raw_pages"] = pdf_pages(b["pdf_path"])
    b["encoding"], s, l = detect_encoding(b["raw_pages"])
    print(f'{b["book_id"]}: {len(b["raw_pages"])} pages | Sinhala chars {s:,} | Latin chars {l:,} -> {b["encoding"]}')
    print("   sample raw text:", re.sub(r"\s+", " ", b["raw_pages"][min(15, len(b["raw_pages"])-1)])[:150])

sinhala_lit_g10_11: 140 pages | Sinhala chars 0 | Latin chars 145,027 -> fm
   sample raw text: ye<,S .sh uyd id.rh fia fldam kue;s jd;fhka ye<,S .sh wgf<dia wlafI!ys‚hla fiakdfjka hqla; je kslauqKdy' flajÜgfhda o ;ukaf.a nia rÊcqrejka is;g kxjd 


4. Convert FM-Abhaya to Unicode

In [7]:
conv = Converter("fm_abhaya")
NOISE = re.compile(r"[\u00AD\u200B\uFEFF]")

def to_unicode(page, encoding):
    if encoding == "fm":
        # In this book the FM glyph '“' is used as an opening quotation mark, but the fm_abhaya table
        # maps it to ර්‍ණ. Replace it BEFORE converting, so real ර්ණ words (වර්ණය, කර්ණ) are untouched.
        page = page.replace("“", "\u0001")
        page = conv.convert(page).replace("\u0001", "“")
    return NOISE.sub("", unicodedata.normalize("NFC", page))

for b in BOOKS:
    if b["encoding"] == "none":
        print(f'{b["book_id"]}: no text layer — skip here, use Document AI OCR'); continue
    b["quote_glyphs"] = sum(p.count("“") for p in b["raw_pages"]) if b["encoding"] == "fm" else 0
    b["pages"] = [to_unicode(p, b["encoding"]) for p in b["raw_pages"]]
    json.dump(b["pages"], open(f'{b["out"]}/{b["book_id"]}_pages_unicode.json', "w", encoding="utf-8"), ensure_ascii=False)
    print(f'{b["book_id"]}: {len(b["pages"])} pages converted | “ glyphs kept as quotes: {b["quote_glyphs"]}')
    print("   sample:", re.sub(r"\s+", " ", b["pages"][min(15, len(b["pages"])-1)])[:150])

sinhala_lit_g10_11: 140 pages converted | “ glyphs kept as quotes: 380
   sample: හැළලී ගිය මහා සාගරය සේ කෝප නමැති වාතයෙන් හැළලී ගිය අටළොස් අක්ෂෞහිණියක් සේනාවෙන් යුක්ත වැ නික්මුණාහ. කේවට්ටයෝ ද තමන්ගේ බස් රජ්ජුරුවන් සිතට නංවා ගත නොහී
